In [1]:
from pathlib import Path
import json
import pandas as pd
import numpy as np

from openai import OpenAI

# -----------------
# Paths
# -----------------
REP_PATH = Path("data/processed/venezuela/posts_repr.parquet")
EVAL_DIR = Path("data/evaluated/ven/hourly")
OUT_PATH = Path("data/evaluated/ven/cluster_summaries.parquet")
OUT_PATH.parent.mkdir(parents=True, exist_ok=True)

# -----------------
# Config
# -----------------
MODEL = "gpt-4.1-mini"  # fast + solid for summarization :contentReference[oaicite:1]{index=1}
MAX_POSTS_PER_CLUSTER = 20
MIN_POSTS_PER_CLUSTER = 4
SEED = 0

client = OpenAI(api_key="sense proj")


# -----------------
# Load representation layer
# -----------------
df_repr = pd.read_parquet(REP_PATH)
df_repr.columns = df_repr.columns.str.strip()
df_repr["post_id"] = df_repr["post_id"].astype(str)

# prefer "text" but fall back if your repr uses "tweet"
if "text" not in df_repr.columns and "tweet" in df_repr.columns:
    df_repr = df_repr.rename(columns={"tweet": "text"})

df_repr = df_repr[["post_id", "text"]].dropna(subset=["text"])
df_repr["text"] = df_repr["text"].astype(str)

rng = np.random.default_rng(SEED)

rows = []

# -----------------
# Iterate hourly window files
# -----------------
for fp in sorted(EVAL_DIR.glob("*.parquet")):
    df_eval = pd.read_parquet(fp)
    df_eval.columns = df_eval.columns.str.strip()

    # normalize
    df_eval["post_id"] = df_eval["post_id"].astype(str).str.strip().str.replace(r"\.0$", "", regex=True)
    if "is_noise" in df_eval.columns:
        df_eval = df_eval[~df_eval["is_noise"]]
    df_eval = df_eval.dropna(subset=["cluster_id"])
    if len(df_eval) == 0:
        continue

    df_eval["cluster_id"] = df_eval["cluster_id"].astype(int)
    window = str(df_eval["window"].iloc[0]) if "window" in df_eval.columns else fp.stem

    df = df_eval.merge(df_repr, on="post_id", how="left").dropna(subset=["text"])
    if len(df) == 0:
        continue

    for cid, g in df.groupby("cluster_id"):
        if len(g) < MIN_POSTS_PER_CLUSTER:
            continue

        # sample representative posts (simple random sample; swap for centroid-based if you want)
        if len(g) > MAX_POSTS_PER_CLUSTER:
            g = g.sample(MAX_POSTS_PER_CLUSTER, random_state=SEED)

        excerpts = []
        for t in g["text"].tolist():
            t = t.replace("\n", " ").strip()
            excerpts.append(t[:500])  # cap per post to control prompt size

        prompt = (
            "You are reducing a cluster of social-media posts into a single narrative summary.\n"
            "Write ONE paragraph (3–5 sentences) describing:\n"
            "- the main claim or storyline,\n"
            "- key actors/objects mentioned,\n"
            "- the implied goal or policy preference (if any),\n"
            "- notable framing (e.g., threat, sovereignty, economy),\n"
            "Avoid quoting more than a short phrase; synthesize.\n\n"
            f"Window: {window}\n"
            f"Cluster ID: {cid}\n\n"
            "Posts:\n- " + "\n- ".join(excerpts)
        )

        resp = client.responses.create(
            model=MODEL,
            input=[{
                "role": "user",
                "content": [{"type": "input_text", "text": prompt}]
            }],
            max_output_tokens=220,
        )
        summary = resp.output_text.strip()

        rows.append({
            "window": window,
            "cluster_id": int(cid),
            "n_posts": int(len(g)),
            "summary": summary,
        })

out = pd.DataFrame(rows).sort_values(["window", "cluster_id"])
out.to_parquet(OUT_PATH, index=False)

print(f"Wrote {len(out)} cluster summaries to {OUT_PATH}")


Wrote 87 cluster summaries to data/evaluated/ven/cluster_summaries.parquet


In [1]:
!pip install -U typing_extensions
!pip install openai


[notice] A new release of pip is available: 24.0 -> 25.3
[notice] To update, run: python -m pip install --upgrade pip

[notice] A new release of pip is available: 24.0 -> 25.3
[notice] To update, run: python -m pip install --upgrade pip


In [10]:
# -------------------------
# Generate short narrative / theme labels for clusters (Jupyter)
# -------------------------

from __future__ import annotations

import json
import os
import random
import time
from pathlib import Path
from typing import Dict, List, Optional

import numpy as np
import pandas as pd
from tqdm.auto import tqdm
from sklearn.metrics.pairwise import cosine_similarity

from openai import OpenAI


# -------------------------
# Config (edit here)
# -------------------------
REP_PATH = Path("data/processed/venezuela/posts_repr.parquet")
EVAL_DIR = Path("data/evaluated/ven/hourly")
SUM_PATH = Path("data/evaluated/ven/cluster_summaries.parquet")  # optional
OUT_PATH = Path("data/evaluated/ven/cluster_labels.parquet")

MODEL = "gpt-4.1-mini"
N_POSTS_FOR_LABEL = 16
MIN_POSTS_PER_CLUSTER = 8
MAX_CHARS_PER_POST = 420

MAX_RETRIES = 6
BASE_SLEEP = 1.5
SEED = 0

random.seed(SEED)
np.random.seed(SEED)


# -------------------------
# Helpers
# -------------------------
def normalize_text(t: str) -> str:
    if t is None:
        return ""
    return " ".join(str(t).replace("\n", " ").split())


def as_np(x) -> np.ndarray:
    if isinstance(x, np.ndarray):
        return x
    return np.array(x, dtype=np.float32)


def pick_central_posts(df_cluster: pd.DataFrame, k: int) -> pd.DataFrame:
    """Pick up to k posts closest to centroid (cosine)."""
    dfc = df_cluster.dropna(subset=["text"]).copy()
    if len(dfc) <= k:
        return dfc

    dfc["text_norm"] = dfc["text"].map(normalize_text).str.lower()
    dfc = dfc.drop_duplicates(subset=["text_norm"])

    if "embedding" not in dfc.columns or dfc["embedding"].isna().all():
        return dfc.sample(n=min(k, len(dfc)), random_state=SEED)

    embs = np.stack(dfc["embedding"].map(as_np).values)
    centroid = embs.mean(axis=0, keepdims=True)
    sims = cosine_similarity(embs, centroid).ravel()

    dfc["sim"] = sims
    return dfc.sort_values("sim", ascending=False).head(k)


def call_openai_label(
    client: OpenAI,
    window: str,
    cluster_id: int,
    posts: List[str],
    summary: Optional[str],
) -> str:
    posts_block = "\n".join([f"- {p}" for p in posts])

    summary_block = ""
    if summary:
        summary_block = f"\nContext summary:\n{summary}\n"

    prompt = f"""
You create concise narrative/theme names for clusters of social media posts.

Constraints:
- 3 to 8 words
- Prefer concrete nouns / actors
- Avoid generic words like "discussion", "debate", "issue"
- No quotes
- No punctuation at the end

Window: {window}
Cluster ID: {cluster_id}
{summary_block}

Representative posts:
{posts_block}

Return ONLY the label.
""".strip()

    last_err = None
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            resp = client.responses.create(
                model=MODEL,
                input=[{
                    "role": "user",
                    "content": [{"type": "input_text", "text": prompt}]
                }],
                max_output_tokens=40,
            )
            label = resp.output_text.strip()
            label = label.split("\n", 1)[0].strip()
            label = label.rstrip(" .,:;!-–—")

            words = label.split()
            if 3 <= len(words) <= 8:
                return label

            if len(words) > 8:
                return " ".join(words[:8])

            raise ValueError(f"Invalid label: {label}")

        except Exception as e:
            last_err = e
            time.sleep(min(BASE_SLEEP * (2 ** (attempt - 1)), 20))

    raise RuntimeError(f"Labeling failed: {last_err}")


# -------------------------
# Load data
# -------------------------
assert REP_PATH.exists(), f"Missing {REP_PATH}"
assert EVAL_DIR.exists(), f"Missing {EVAL_DIR}"

client = OpenAI(api_key="")
 # requires OPENAI_API_KEY

df_repr = pd.read_parquet(REP_PATH)
df_repr.columns = df_repr.columns.str.strip()

if "text" not in df_repr.columns and "tweet" in df_repr.columns:
    df_repr = df_repr.rename(columns={"tweet": "text"})

df_repr["post_id"] = df_repr["post_id"].astype(str).str.replace(r"\.0$", "", regex=True)

keep_cols = ["post_id", "text"]
if "embedding" in df_repr.columns:
    df_repr["embedding"] = df_repr["embedding"].map(as_np)
    keep_cols.append("embedding")

df_repr = df_repr[keep_cols].dropna(subset=["text"])

# Optional summaries
summary_map: Dict[str, str] = {}
if SUM_PATH.exists():
    df_sum = pd.read_parquet(SUM_PATH)
    df_sum.columns = df_sum.columns.str.strip()
    df_sum["window"] = df_sum["window"].astype(str)
    df_sum["cluster_id"] = df_sum["cluster_id"].astype(int)

    summary_map = {
        f"{r.window}|{r.cluster_id}": str(r.summary)
        for r in df_sum.itertuples(index=False)
    }


# -------------------------
# Main loop
# -------------------------
results = []

eval_files = sorted(EVAL_DIR.glob("*.parquet"))

for fp in tqdm(eval_files, desc="Windows"):
    df_eval = pd.read_parquet(fp)
    df_eval.columns = df_eval.columns.str.strip()

    if "window" not in df_eval.columns:
        df_eval["window"] = fp.stem

    df_eval["post_id"] = df_eval["post_id"].astype(str).str.replace(r"\.0$", "", regex=True)

    if "is_noise" in df_eval.columns:
        df_eval = df_eval[~df_eval["is_noise"]]

    df_eval = df_eval.dropna(subset=["cluster_id"])
    if df_eval.empty:
        continue

    df_eval["cluster_id"] = df_eval["cluster_id"].astype(int)
    window = str(df_eval["window"].iloc[0])

    df = df_eval.merge(df_repr, on="post_id", how="left").dropna(subset=["text"])
    if df.empty:
        continue

    for cid, g in df.groupby("cluster_id"):
        if len(g) < MIN_POSTS_PER_CLUSTER:
            continue

        reps = pick_central_posts(g, N_POSTS_FOR_LABEL)
        posts = [
            normalize_text(t)[:MAX_CHARS_PER_POST]
            for t in reps["text"].tolist()
            if normalize_text(t)
        ]

        if len(posts) < 3:
            continue

        key = f"{window}|{cid}"
        summary = summary_map.get(key)

        label = call_openai_label(
            client,
            window=window,
            cluster_id=cid,
            posts=posts,
            summary=summary,
        )

        results.append({
            "window": window,
            "cluster_id": cid,
            "n_posts": len(g),
            "label": label,
            "model": MODEL,
        })


# -------------------------
# Save output
# -------------------------
df_out = pd.DataFrame(results).sort_values(["window", "cluster_id"]).reset_index(drop=True)
df_out.to_parquet(OUT_PATH, index=False)

df_out.head(), f"Wrote {len(df_out)} labels → {OUT_PATH}"





Windows:   0%|          | 0/29 [00:00<?, ?it/s]


Windows:  14%|█▍        | 4/29 [00:02<00:13,  1.88it/s]


Windows:  17%|█▋        | 5/29 [00:05<00:33,  1.39s/it]


Windows:  21%|██        | 6/29 [00:08<00:38,  1.68s/it]


Windows:  24%|██▍       | 7/29 [00:11<00:43,  1.98s/it]


Windows:  28%|██▊       | 8/29 [00:14<00:50,  2.41s/it]


Windows:  31%|███       | 9/29 [00:17<00:48,  2.42s/it]


Windows:  34%|███▍      | 10/29 [00:19<00:48,  2.53s/it]


Windows:  38%|███▊      | 11/29 [00:21<00:41,  2.33s/it]


Windows:  41%|████▏     | 12/29 [00:23<00:39,  2.31s/it]


Windows:  45%|████▍     | 13/29 [00:25<00:32,  2.03s/it]


Windows:  48%|████▊     | 14/29 [00:26<00:27,  1.85s/it]


Windows:  52%|█████▏    | 15/29 [00:29<00:27,  1.98s/it]


Windows:  55%|█████▌    | 16/29 [00:30<00:25,  1.93s/it]


Windows:  59%|█████▊    | 17/29 [00:32<00:21,  1.80s/it]


Windows:  62%|██████▏   | 18/29 [00:33<00:18,  1.68s/it]


Windows:  66%|██████▌   | 19/29 [00:36<00:20,  2.09s/it]


Windows: 

(          window  cluster_id  n_posts  \
 0  2026-01-02-14           0       19   
 1  2026-01-02-14           1       13   
 2  2026-01-02-18           0       30   
 3  2026-01-02-18           1       11   
 4  2026-01-02-22           0      363   
 
                                                label         model  
 0                 Maduro Regime and US Oil Conflicts  gpt-4.1-mini  
 1                     Maduro Arrest and US Detention  gpt-4.1-mini  
 2   US Military Captures Venezuelan President Maduro  gpt-4.1-mini  
 3     Venezuelan Leadership Crisis and Proof of Life  gpt-4.1-mini  
 4  Trump Administration Captures Maduro in Venezuela  gpt-4.1-mini  ,
 'Wrote 87 labels → data/evaluated/ven/cluster_labels.parquet')

In [6]:
!pip install iprogress


[notice] A new release of pip is available: 24.0 -> 25.3
[notice] To update, run: python -m pip install --upgrade pip
